In [1]:
clean_up = True # if True, remove all gams related files from working folder before starting
%run stdPackages.ipynb

The file _gams_py_gdb0.gdx is still active and was not deleted.
The file _gams_py_gdb1.gdx is still active and was not deleted.


*Load specific modules used here:*

In [2]:
os.chdir(main)
%run stdPlotting.ipynb
os.chdir(d['py'])
import mCGE

# Shocks

In [3]:
slides = True
tPlot_1 = 2030 # if one year in plot, use this
tPlot = pd.Index(range(2019, 2034), name = 't')

Load base model and data:

In [4]:
t0 = 2019
name = f'vGRSIntRC{t0}CGE'
M = mCGE.WasteManagementCGE.load(os.path.join(d['data'], name)) # load model
ws = M.ws 
db0 = M.db.copy() # store initial solution

## TESTS:

*A few definitions:*

In [5]:
vm = adj.rc_pd(M.db('n2m2k2o'), pd.Index(['V'], name = 'k')).get_level_values('n') # virgin materials
rm = adj.rc_pd(M.db('n2m2k2o'), pd.Index(['R'], name = 'k')).get_level_values('n') # recycled materials
n2m = M.db('n2m2k2o').droplevel(['k','o'])
sSum = M.db('s').difference(M.db('s_f')) 

*Auxiliary functions used throughout:*

In [6]:
def f_Qv(dbi, vmi = vm, si = sSum):
    """ Virgin material demand """
    return adjMultiIndex.applyMult(adj.rc_pd(dbi('qD'), ('and', [vmi, si])).groupby(['t','n']).sum(), n2m).groupby(['t','m']).sum()
def f_Qr(dbi, rmi = rm, si = sSum):
    """ Recycled material demand"""
    return adjMultiIndex.applyMult(adj.rc_pd(dbi('qD'), ('and', [rmi, si])).groupby(['t','n']).sum(), n2m).groupby(['t','m']).sum()
def f_Qt(dbi, vmi = vm, rmi = rm, si = sSum):
    """ Total material demand"""
    return f_Qv(dbi,vmi=vmi, si = si).add(f_Qr(dbi, rmi=rmi, si = si), fill_value=0)

def f_Rbv(di, d0):
    """ Rebound effect, virgin materials"""
    return ((di['Δexp']-(di['Qv']-d0['Qv']))/di['Δexp']).dropna()
def f_Rbr(di,d0):
    """ ΔRecycled materials/mechanical effect of shock"""
    return ((di['Δexp']-(di['Qr']-d0['Qr']))/di['Δexp']).dropna()
def f_Rbt(di,d0):
    """ ΔMaterials/mechanical effect of shock"""
    return ((di['Δexp']-(di['Qt']-d0['Qt']))/di['Δexp']).dropna()
def f_ΔQv_Qv(di,d0):
    """ Relative change in virgin material consumption """
    return ((di['Qv']-d0['Qv'])/d0['Qv']).dropna()
def f_ΔQr_Qr(di,d0):
    """ Relative change in recycled material consumption """
    return ((di['Qr']-d0['Qr'])/d0['Qr']).dropna()
def f_ΔQt_Qt(di,d0):
    """ Relative change in total material consumption """
    return ((di['Qt']-d0['Qt'])/d0['Qt']).dropna()
def f_CMUR(di):
    """ Circular material use rate"""
    return di['Qr']/di['Qt']

# Repeat but with summing over $m$:
def f_Qv_sum(di):
    """ Virgin material demand, sum over $m$ """
    return di['Qv'].groupby('t').sum()
def f_Qr_sum(di):
    """ Recycled material demand, sum over $m$ """
    return di['Qr'].groupby('t').sum()
def f_Qt_sum(di):
    """ Total material demand, sum over $m$ """
    return di['Qt'].groupby('t').sum()
def f_Rbv_sum(di, d0):
    """ Rebound effect, virgin materials"""
    return ((di['Δexp'].groupby('t').sum()-(di['Qv_sum']-d0['Qv_sum']))/di['Δexp'].groupby('t').sum()).dropna()
def f_ΔQv_Qv_sum(di,d0):
    """ Relative change in virgin material consumption """
    return ((di['Qv_sum']-d0['Qv_sum'])/d0['Qv_sum']).dropna()
def f_ΔQr_Qr_sum(di,d0):
    """ Relative change in recycled material consumption """
    return ((di['Qr_sum']-d0['Qr_sum'])/d0['Qr_sum']).dropna()
def f_ΔQt_Qt_sum(di,d0):
    """ Relative change in total material consumption """
    return ((di['Qt_sum']-d0['Qt_sum'])/d0['Qt_sum']).dropna()
def f_CMUR_sum(di):
    """ Circular material use rate"""
    return di['Qr_sum']/di['Qt_sum']

In [7]:
def fReport0(d0, db0, vmi = vm, rmi = rm, si = sSum):
    d0['db'] = db0
    d0['Qv'] = f_Qv(db0, vmi = vmi, si = si)
    d0['Qr'] = f_Qr(db0, rmi = rmi, si = si)
    d0['Qt'] = f_Qt(db0, vmi = vmi, rmi = rmi, si = si)
    d0['CMUR'] = f_CMUR(d0)
    d0['Qv_sum'] = f_Qv_sum(d0)
    d0['Qr_sum'] = f_Qr_sum(d0)
    d0['Qt_sum'] = f_Qt_sum(d0)
    d0['CMUR_sum'] = f_CMUR_sum(d0)
    return d0

In [8]:
def fReport(di, d0, dbi, vmi = vm, rmi = rm, si = sSum):
    di['db'] = dbi
    di['Qv'] = f_Qv(dbi, vmi = vmi, si = si)
    di['Qr'] = f_Qr(dbi, rmi = rmi, si = si)
    di['Qt'] = f_Qt(dbi, vmi = vmi, rmi = rmi, si =si)
    di['Rbv'] = f_Rbv(di, d0)
    di['Rbr'] = f_Rbr(di, d0)
    di['Rbt'] = f_Rbt(di, d0)
    di['ΔQv/Qv'] = f_ΔQv_Qv(di,d0)
    di['ΔQr/Qr'] = f_ΔQr_Qr(di,d0)
    di['ΔQt/Qt'] = f_ΔQt_Qt(di,d0)
    di['CMUR'] = f_CMUR(di)

    di['Qv_sum'] = f_Qv_sum(di)
    di['Qr_sum'] = f_Qr_sum(di)
    di['Qt_sum'] = f_Qt_sum(di)
    di['Rbv_sum'] = f_Rbv_sum(di, d0)
    di['ΔQv/Qv_sum'] = f_ΔQv_Qv_sum(di,d0)
    di['ΔQr/Qr_sum'] = f_ΔQr_Qr_sum(di,d0)
    di['ΔQt/Qt_sum'] = f_ΔQt_Qt_sum(di,d0)
    di['CMUR_sum'] = f_CMUR_sum(di)
    return di

## 1. Models

To decompose the effects of various shocks, we create two additional versions:
* Version A: One with production firms, investment firms, households.
* Version B: Version A + waste treatment.

In [9]:
M_A = mCGE.WasteManagementCGE(name = 'PIC', database = M.db, compiler = M.compiler) # Production, investment, consumption
M_B = mCGE.WasteManagementCGE(name = 'PICW', database = M.db, compiler= M.compiler) # Production, investment, consumption, waste management

Add modules (refer to instances from main module):

In [10]:
M_A.m = {k:v for k,v in M.m.items() if k in ('P','I','C')}
M_B.m = {k:v for k,v in M.m.items() if k in ('P','I','C','W')}

[mi.__setattr__('partial',True) for mi in M_B.m.values()]; # specify partial equilibrium model, ...
[mi.initGroups() for mi in M_B.m.values()]; # re-initialize groups with partial setting

Baseline solution:

In [11]:
d0 = fReport0({}, M.db.copy())

## 2. Increase in the internal recycling rate

In [12]:
shocks = {}

### A. Metal only

Define shock:

In [13]:
def exp_IRRmetal(Mi, solDict):
    ξnew = adj.rc_pd(Mi.db('intRcEff'), pd.Index(['Metal'],name='m'))+0.1
    μWP0 = (Mi.db('uWS_int') * (Mi.db('qWS')/(1-Mi.db('uWS_int') * Mi.db('intRcEff')))).dropna() # μint * W^P in baseline
    solDict['shock'] = ξnew
    solDict['Δexp'] = -((ξnew-Mi.db('intRcEff')) * μWP0).dropna().groupby(['t','m']).sum() # expected decrease in use of virgin materials
    return solDict

Add solution dictionary:

In [14]:
name = 'IRRmetal'
shocks[name] = dict.fromkeys(['A','B','F']) # two partial models and full
shocks_i = shocks[name]
for k in shocks_i:
    shocks_i[k] = exp_IRRmetal(M, {})

Solve all three models:

In [15]:
M.db.aom(shocks_i['F']['shock'], priority = 'second')
M.db.mergeInternal()
shocks_i['A']['db'] = M_A.solve(state = 'B')
shocks_i['B']['db'] = M_B.solve(state = 'B')
shocks_i['F']['db'] = M.solve(state = 'B')

Report for each:

In [16]:
for k, di in shocks_i.items():
    di = fReport(di, d0, di['db'])

Revert model data

In [17]:
M.db.aom(adj.rc_pd(d0['db'](shocks_i['F']['shock'].name), shocks_i['F']['shock']), priority = 'second')

### B. Plastic only

Define shock:

In [18]:
def exp_IRRplastic(Mi, solDict):
    ξnew = adj.rc_pd(Mi.db('intRcEff'), pd.Index(['Plastic'],name='m'))+0.1
    μWP0 = (Mi.db('uWS_int') * (Mi.db('qWS')/(1-Mi.db('uWS_int') * Mi.db('intRcEff')))).dropna() # μint * W^P in baseline
    solDict['shock'] = ξnew
    solDict['Δexp'] = -((ξnew-Mi.db('intRcEff')) * μWP0).dropna().groupby(['t','m']).sum() # expected decrease in use of virgin materials
    return solDict

Add solution dictionary:

In [19]:
name = 'IRRplastic'
shocks[name] = dict.fromkeys(['A','B','F']) # two partial models and full
shocks_i = shocks[name]
for k in shocks_i:
    shocks_i[k] = exp_IRRplastic(M, {})

Solve all three models:

In [20]:
M.db.aom(shocks_i['F']['shock'], priority = 'second')
M.db.mergeInternal()
shocks_i['A']['db'] = M_A.solve(state = 'B')
shocks_i['B']['db'] = M_B.solve(state = 'B')
shocks_i['F']['db'] = M.solve(state = 'B')

Report for each:

In [21]:
for k, di in shocks_i.items():
    di = fReport(di, d0, di['db'])

Revert model data

In [22]:
M.db.aom(adj.rc_pd(d0['db'](shocks_i['F']['shock'].name), shocks_i['F']['shock']), priority = 'second')

### C. Shock to all

In [23]:
def exp_IRRall(Mi, solDict):
    ξnew = Mi.db('intRcEff')+0.1
    μWP0 = (Mi.db('uWS_int') * (Mi.db('qWS')/(1-Mi.db('uWS_int') * Mi.db('intRcEff')))).dropna() # μint * W^P in baseline
    solDict['shock'] = ξnew
    solDict['Δexp'] = -((ξnew-Mi.db('intRcEff')) * μWP0).dropna().groupby(['t','m']).sum() # expected decrease in use of virgin materials
    return solDict

Add solution dictionary:

In [24]:
name = 'IRRall'
shocks[name] = dict.fromkeys(['A','B','F']) # two partial models and full
shocks_i = shocks[name]
for k in shocks_i:
    shocks_i[k] = exp_IRRall(M, {})

Solve all three models:

In [25]:
M.db.aom(shocks_i['F']['shock'], priority = 'second')
M.db.mergeInternal()
shocks_i['A']['db'] = M_A.solve(state = 'B')
shocks_i['B']['db'] = M_B.solve(state = 'B')
shocks_i['F']['db'] = M.solve(state = 'B')

Report for each:

In [26]:
for k, di in shocks_i.items():
    di = fReport(di, d0, di['db'])

Revert model data

In [27]:
M.db.aom(adj.rc_pd(d0['db'](shocks_i['F']['shock'].name), shocks_i['F']['shock']), priority = 'second')

### D. Collect and plot results

**Rebound rate over time -- across all materials**

In [28]:
%%capture
x = 'IRRall'
di = shocks[x]['F']
largeFont()
df = adj.rc_pd(di['Rbv_sum'], tPlot)
if slides:
    fig, ax = plt.subplots(1,1,figsize = (14,10))
else:
    fig, ax = plt.subplots(1,1,figsize = (8,5))
seaborn.lineplot(data = df, linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('Rebound rate');
ax.axhline(1, color = 'k', linewidth = 1, alpha = .5);
# ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{int(x)}'))
fig.tight_layout();
if slides:
    fig.savefig(os.path.join(d['figs'],f"GRSIntRC_{x}sum_Rbv_slides.png"),facecolor='#FAFAFA',edgecolor='k')
else:
    fig.savefig(os.path.join(d['figs'],f"GRSIntRC_{x}sum_Rbv.pdf"),edgecolor='k')

**Change in material throughout - across all materials:**

In [29]:
%%capture
largeFont()
x = 'IRRall'
di = shocks[x]['F']
df = pd.concat([di[f'ΔQ{k}/Q{k}_sum'].rename(f'$ΔQ_{k}/Q_{k}$') for k in ('v','r','t')], axis = 1)*100
df = adj.rc_pd(df, tPlot)
if slides:
    fig, ax = plt.subplots(1,1,figsize = (14,10))
else:
    fig, ax = plt.subplots(1,1,figsize = (8,5))
seaborn.lineplot(data = df, linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5);
fig.tight_layout();
if slides:
    fig.savefig(os.path.join(d['figs'],f"GRSIntRC_{x}sum_DiffMC_slides.png"),facecolor='#FAFAFA',edgecolor='k')
else:
    fig.savefig(os.path.join(d['figs'],f"GRSIntRC_{x}sum_DiffMC.pdf"),edgecolor='k')

**Rebound rate over time - different material types:**

In [30]:
%%capture
largeFont()
x = 'IRRall'
di = shocks[x]['F']
df = adj.rc_pd(di['Rbv'], tPlot).unstack('m').rename_axis(columns = None)
if slides:
    fig, ax = plt.subplots(1,1,figsize = (14,10))
else:
    fig, ax = plt.subplots(1,1,figsize = (8,5))
seaborn.lineplot(data = df, linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('Rebound rate');
ax.axhline(1, color = 'k', linewidth = 1, alpha = .5);
fig.tight_layout();
if slides:
    fig.savefig(os.path.join(d['figs'],f"GRSIntRC_{x}t_Rbv_slides.png"),facecolor='#FAFAFA',edgecolor='k')
else:
    fig.savefig(os.path.join(d['figs'],f"GRSIntRC_{x}t_Rbv.pdf"),edgecolor='k')

**Change in material throughout - different material types:**

In [31]:
%%capture
largeFont()
x = 'IRRall'
ti = tPlot_1
di = shocks[x]['F']
df = pd.concat([di['ΔQv/Qv'].xs(ti), di['ΔQr/Qr'].xs(ti), di['ΔQt/Qt'].xs(ti), di['CMUR'].xs(ti)-d0['CMUR'].xs(ti)], axis = 1).set_axis(['$ΔQ_v/Q_v$',
                                                                                                                                    '$ΔQ_r/Q_r$',
                                                                                                                                    '$ΔQ_t/Q_t$',
                                                                                                                                    '$ΔCMUR$'], axis = 1) * 100
if slides:
    fig, ax = plt.subplots(1,1,figsize = (14,10))
else:
    fig, ax = plt.subplots(1,1,figsize = (8,5))
df[['$ΔQ_v/Q_v$','$ΔQ_r/Q_r$','$ΔQ_t/Q_t$']].plot.bar(ax = ax)
ax.set_xlabel('');
ax.set_ylabel('% change');
fig.tight_layout();
if slides:
    fig.savefig(os.path.join(d['figs'],f"GRSIntRC_{x}_DiffMC_slides.png"),facecolor='#FAFAFA',edgecolor='k')
else:
    fig.savefig(os.path.join(d['figs'],f"GRSIntRC_{x}_DiffMC.pdf"),edgecolor='k')

**Plot change in CMUR:**

In [32]:
%%capture
largeFont()
if slides:
    fig, ax = plt.subplots(1,1,figsize = (14,10))
else:
    fig, ax = plt.subplots(1,1,figsize = (8,5))
df['$ΔCMUR$'].plot.bar(ax=ax)
ax.set_xlabel('');
ax.set_ylabel('$Δ CMUR$');
fig.tight_layout();
if slides:
    fig.savefig(os.path.join(d['figs'],f"GRSIntRC_{x}_CMUR_slides.png"),facecolor='#FAFAFA',edgecolor='k')
else:
    fig.savefig(os.path.join(d['figs'],f"GRSIntRC_{x}_CMUR.pdf"),edgecolor='k')

**Decomposition of the three effects:**

In [33]:
%%capture
largeFont()
x = 'IRRall'
df = adj.rc_pd(pd.concat([shocks[x][k]['Rbv'].rename(k) for k in ('A','B','F')], axis = 1), tPlot)
df.columns = df.columns.map({'A': 'Partial', 'B': 'Partial w. waste', 'F': 'Full'})
fig, axes = plt.subplots(1, 3, figsize=(14,4));
for i in range(3):
    ax = plt.subplot(1,3,i+1)
    j = df.index.levels[-1][i] # material type
    dfi = df.xs(j, level = 'm')
    if i == 0:
        seaborn.lineplot(data = dfi, linewidth = 3, ax = ax, legend = 'full');
        handles, labels = ax.get_legend_handles_labels()
        ax.get_legend().remove()
    else:
        seaborn.lineplot(data = dfi, linewidth = 3, ax = ax, legend = False);
    ax.set_xlabel('');
    ax.set_ylabel('Rebound rate');
    ax.axhline(1, color = 'k', linewidth = 1, alpha = .5);
fig.legend(handles, labels, loc = 8, ncol = 3, fontsize = 16, frameon = True)
fig.tight_layout();
fig.subplots_adjust(bottom=0.25)
if slides:
    fig.savefig(os.path.join(d['figs'],f"GRSIntRC_{x}t_Rbv_Decomp_slides.png"),facecolor='#FAFAFA',edgecolor='k')
else:
    fig.savefig(os.path.join(d['figs'],f"GRSIntRC_{x}t_Rbv_Decomp.pdf"),edgecolor='k')

**Compare rebound rates with single material shock and total**

In [34]:
%%capture
largeFont()
if slides:
    fig, ax = plt.subplots(1,1,figsize = (14,10))
else:
    fig, ax = plt.subplots(1,1,figsize = (8,5))
# Metal
dfMetal = adj.rc_pd( pd.concat([shocks[x]['F']['Rbv'].xs('Metal',level='m') for x in ('IRRall','IRRmetal')], axis = 1).set_axis(['All','Metal only'], axis =1), tPlot)
seaborn.lineplot(data = dfMetal, linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('Rebound rate');
ax.axhline(1, color = 'k', linewidth = 1, alpha = .5);
fig.tight_layout();
if slides:
    fig.savefig(os.path.join(d['figs'],f"GRSIntRC_IRR_MetalVsAll_slides.png"),facecolor='#FAFAFA',edgecolor='k')
else:
    fig.savefig(os.path.join(d['figs'],f"GRSIntRC_IRR_MetalVsAll.pdf"),edgecolor='k')
# Plastic
if slides:
    fig, ax = plt.subplots(1,1,figsize = (14,10))
else:
    fig, ax = plt.subplots(1,1,figsize = (8,5))
# Metal
dfPlastic = adj.rc_pd(pd.concat([shocks[x]['F']['Rbv'].xs('Plastic',level='m') for x in ('IRRall','IRRplastic')], axis = 1).set_axis(['All','Plastic only'], axis =1), tPlot)
seaborn.lineplot(data = dfPlastic, linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('Rebound rate');
ax.axhline(1, color = 'k', linewidth = 1, alpha = .5);
fig.tight_layout();
if slides:
    fig.savefig(os.path.join(d['figs'],f"GRSIntRC_IRR_PlasticVsAll_slides.png"),facecolor='#FAFAFA',edgecolor='k')
else:
    fig.savefig(os.path.join(d['figs'],f"GRSIntRC_IRR_PlasticVsAll.pdf"),edgecolor='k')

## 3. Changes in recycling technology in waste management

In [35]:
def f_intApprox(x, l = 0, u = 1, eps = 0.0001):
    return (l+np.sqrt(np.square(x-l)+eps)+u-np.sqrt(np.square(x-u)+eps))/2
def f_techGInv(y, l = 0, u = 1, b = 2):
    return ((u-y)/(u-l)).pow(1/b)
def f_techF(r, l = 0, u = 1, b = 2):
    return l+(u-l)*(1-r.pow(b)/(1+b))

### A. Define shock and compute mechanical effect

In [36]:
name = 'RCEff' # Recycling efficiency 
shocks[name] = {}
di = shocks[name] 

Increase $\underline{\alpha}$ in recycling technology for metal and compute mechanical effect of shock:

In [37]:
new_αl = M.db('WTD_alphal')+0.1
# new_αl = adj.rc_pd(M.db('WTD_alphal'), pd.Index(['Metal'], name = 'm'))+0.1
di['shock']= new_αl
M.db.aom(di['shock'], priority = 'second')
M.db.mergeInternal()

Given prices and scale (amount of waste to be treated), how much extra recycling do we get out of this shock?

In [38]:
l, u, b, eps = M.db('WTD_alphal'), M.db('WTD_alphau'), M.db('WTD_beta'), M.db('WTD_smooth')
y = f_intApprox(M.db('WTD_zetaE'), l = l, u = u, eps = eps) 
r = f_techGInv(y, l= l, u = u, b = b) # R/R̄
a = f_techF(r, l = l, u = u, b = b) # A/R
R = M.db('WTD_W') * r # R
A = (a * R).dropna() # A

What is the reduction in virgin materials compared to baseline, if they reduce this 1-to-1:

In [39]:
qSR0 = adjMultiIndex.applyMult(d0['db']('qS').xs('Waste',level='s'), db0('nr2m_D')).droplevel('n').dropna() # supply of recycled materials in baseline, mapped to 'm' index
di['Δexp'] = -(A-qSR0).dropna()
# di['Δexp'] = -adj.rc_pd((A-qSR0).dropna(), pd.Index(['Metal'], name = 'm'))

Solve:

In [40]:
di['db'] = M.solve(state = 'B')

Get results:

In [41]:
di = fReport(di, d0, di['db'])

Revert model data:

In [42]:
M.db.aom(adj.rc_pd(d0['db'](di['shock'].name), di['shock']))

### B. Collect and plot results

**Rebound rate over time -- across all materials**

In [43]:
%%capture
x = 'RCEff'
di = shocks[x]
largeFont()
df = adj.rc_pd(di['Rbv_sum'], tPlot)
if slides:
    fig, ax = plt.subplots(1,1,figsize = (14,10))
else:
    fig, ax = plt.subplots(1,1,figsize = (8,5))
seaborn.lineplot(data = df, linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('Rebound rate');
ax.axhline(1, color = 'k', linewidth = 1, alpha = .5);
fig.tight_layout();
if slides:
    fig.savefig(os.path.join(d['figs'],f"GRSIntRC_{x}sum_Rbv_slides.png"),facecolor='#FAFAFA',edgecolor='k')
else:
    fig.savefig(os.path.join(d['figs'],f"GRSIntRC_{x}sum_Rbv.pdf"),edgecolor='k')

**Change in material throughout - across all materials:**

In [44]:
%%capture
largeFont()
x = 'RCEff'
di = shocks[x]
df = pd.concat([di[f'ΔQ{k}/Q{k}_sum'].rename(f'$ΔQ_{k}/Q_{k}$') for k in ('v','r','t')], axis = 1)*100
df = adj.rc_pd(df, tPlot)
if slides:
    fig, ax = plt.subplots(1,1,figsize = (14,10))
else:
    fig, ax = plt.subplots(1,1,figsize = (8,5))
seaborn.lineplot(data = df, linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5);
fig.tight_layout();
if slides:
    fig.savefig(os.path.join(d['figs'],f"GRSIntRC_{x}sum_DiffMC_slides.png"),facecolor='#FAFAFA',edgecolor='k')
else:
    fig.savefig(os.path.join(d['figs'],f"GRSIntRC_{x}sum_DiffMC.pdf"),edgecolor='k')

**Rebound rate over time - different material types:**

In [45]:
%%capture
largeFont()
x = 'RCEff'
di = shocks[x]
df = adj.rc_pd(di['Rbv'], tPlot).unstack('m').rename_axis(columns = None)
if slides:
    fig, ax = plt.subplots(1,1,figsize = (14,10))
else:
    fig, ax = plt.subplots(1,1,figsize = (8,5))
seaborn.lineplot(data = df, linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('Rebound rate');
ax.axhline(1, color = 'k', linewidth = 1, alpha = .5);
fig.tight_layout();
if slides:
    fig.savefig(os.path.join(d['figs'],f"GRSIntRC_{x}t_Rbv_slides.png"),facecolor='#FAFAFA',edgecolor='k')
else:
    fig.savefig(os.path.join(d['figs'],f"GRSIntRC_{x}t_Rbv.pdf"),edgecolor='k')

**Change in material throughout - different material types:**

In [46]:
%%capture
largeFont()
x = 'RCEff'
ti = tPlot_1
di = shocks[x]
df = pd.concat([di['ΔQv/Qv'].xs(ti), di['ΔQr/Qr'].xs(ti), di['ΔQt/Qt'].xs(ti), di['CMUR'].xs(ti)-d0['CMUR'].xs(ti)], axis = 1).set_axis(['$ΔQ_v/Q_v$',
                                                                                                                                    '$ΔQ_r/Q_r$',
                                                                                                                                    '$ΔQ_t/Q_t$',
                                                                                                                                    '$ΔCMUR$'], axis = 1) * 100
if slides:
    fig, ax = plt.subplots(1,1,figsize = (14,10))
else:
    fig, ax = plt.subplots(1,1,figsize = (8,5))
df[['$ΔQ_v/Q_v$','$ΔQ_r/Q_r$','$ΔQ_t/Q_t$']].plot.bar(ax = ax)
ax.set_xlabel('');
ax.set_ylabel('% change');
fig.tight_layout();
if slides:
    fig.savefig(os.path.join(d['figs'],f"GRSIntRC_{x}_DiffMC_slides.png"),facecolor='#FAFAFA',edgecolor='k')
else:
    fig.savefig(os.path.join(d['figs'],f"GRSIntRC_{x}_DiffMC.pdf"),edgecolor='k')

**Plot change in CMUR:**

In [47]:
%%capture
largeFont()
if slides:
    fig, ax = plt.subplots(1,1,figsize = (14,10))
else:
    fig, ax = plt.subplots(1,1,figsize = (8,5))
df['$ΔCMUR$'].plot.bar(ax=ax)
ax.set_xlabel('');
ax.set_ylabel('$Δ CMUR$');
fig.tight_layout();
if slides:
    fig.savefig(os.path.join(d['figs'],f"GRSIntRC_{x}_CMUR_slides.png"),facecolor='#FAFAFA',edgecolor='k')
else:
    fig.savefig(os.path.join(d['figs'],f"GRSIntRC_{x}_CMUR.pdf"),edgecolor='k')

## 4. Tax on virgin materials

In [48]:
name = 'taxVirgin' # Recycling efficiency 
shocks[name] = {}
di = shocks[name] 

Shock:

In [49]:
di['shock'] = adj.rc_pd(M.db('tauD'), vm)+0.1
M.db.aom(di['shock'], priority = 'second')
M.db.mergeInternal()
di['db'] = M.solve(state = 'B')

*Note: No "expected" reduction in this case.*

In [50]:
di['Δexp'] = pd.Series(1, index = db0('t')) # add this to avoid errors in standard reporting function
di = fReport(di, d0, di['db'])

Revert model data:

In [51]:
M.db.aom(adj.rc_pd(d0['db'](di['shock'].name), di['shock']))

### B. Plot 

**Change in material consumption in 2030:**

Plot: Relative change in virgin consumption, recycled consumption, total consumption, and change in CMUR

In [52]:
ti = 2030
di = shocks['taxVirgin']
df = pd.concat([di['ΔQv/Qv'].xs(ti), di['ΔQr/Qr'].xs(ti), di['ΔQt/Qt'].xs(ti), di['CMUR'].xs(ti)-d0['CMUR'].xs(ti)], axis = 1).set_axis(['$ΔQ_v/Q_v$',
                                                                                                                                    '$ΔQ_r/Q_r$',
                                                                                                                                    '$ΔQ_t/Q_t$',
                                                                                                                                    '$ΔCMUR$'], axis = 1)

In [53]:
%%capture
# %%capture --no-display
largeFont()
if slides:
    fig, ax = plt.subplots(1,1,figsize = (14,10))
else:
    fig, ax = plt.subplots(1,1,figsize = (8,5))
df[['$ΔQ_v/Q_v$','$ΔQ_r/Q_r$','$ΔQ_t/Q_t$']].plot.bar(ax = ax)
ax.set_xlabel('');
ax.set_ylabel('% change');
fig.tight_layout();
if slides:
    fig.savefig(os.path.join(d['figs'],"GRSIntRC_TaxV_DiffMC_slides.png"),facecolor='#FAFAFA',edgecolor='k')
else:
    fig.savefig(os.path.join(d['figs'],"GRSIntRC_TaxV_DiffMC.pdf"),edgecolor='k')

**Plot change in CMUR:**

In [54]:
%%capture
largeFont()
if slides:
    fig, ax = plt.subplots(1,1,figsize = (14,10))
else:
    fig, ax = plt.subplots(1,1,figsize = (8,5))
df['$ΔCMUR$'].plot.bar(ax = ax)
ax.set_xlabel('');
ax.set_ylabel('$ΔCMUR$');
fig.tight_layout();
if slides:
    fig.savefig(os.path.join(d['figs'],"GRSIntRC_TaxV_DiffCMUR_slides.png"),facecolor='#FAFAFA',edgecolor='k')
else:
    fig.savefig(os.path.join(d['figs'],"GRSIntRC_TaxV_DiffCMUR.pdf"),edgecolor='k')

## 5. Subsidy to recycling/mandate to increase recycling

### A. Define shock and compute mechanical effect

In [55]:
name = 'RCmandate' # Recycling efficiency 
shocks[name] = {}
di = shocks[name] 

Assume that recycling of plastic increases from 42.5% to 50%. The expected effect of this is given by:

In [56]:
targetRate = 0.5
mi = pd.Index(['Plastic'], name = 'm')
sIdx = pd.Index(['Waste'],name='s') # relevant industry
ni = adjMultiIndex.applyMult(mi, M.db('nr2m_D')) 
R̄ = adj.rc_pd((M.db('WTD_W')*(1-M.db('WTD_dmin'))).dropna(), mi)
Rtarget = R̄ * targetRate
di['Δexp'] = (R̄ * adj.rc_pd(M.db('WTD_r'), mi)).dropna()-Rtarget

Compute subsidy that results in this with partial equilibrium assumptions:

In [57]:
pzw = adj.rc_pd(M.db('pD'), M.get('n_ZW',m='W')).droplevel(('s','n'))
pmr = adj.rc_pd(M.db('p'), ni)
pwe = adj.rc_pd(M.db('pS'), M.db('n_wasteE')).droplevel(('s','n'))
Δγ  = adj.rc_pd(M.db('WTD_gr'), mi)-adj.rc_pd(M.db('WTD_ge'), mi)
ζtarget = adj.rc_pd(M.db('WTD_alphau') - (M.db('WTD_alphau')-M.db('WTD_alphal'))*(targetRate**(M.db('WTD_beta'))), mi).dropna()
τ = adjMultiIndex.bc(adj.rc_pd(1/(1+M.db('markup')), sIdx), mi)-(pzw*Δγ+adj.rc_pd(M.db('WTD_gwe'), mi) * adjMultiIndex.bc(pwe, mi))/(pmr*ζtarget)
τ = stdSort(τ.dropna()).droplevel('m').rename('tauS')

Add shock and solve:

In [58]:
di['shock'] = τ
M.db.aom(di['shock'], priority = 'second')
M.db.mergeInternal()
di['db'] = M.solve(state = 'B')

Get results:

In [59]:
di = fReport(di, d0, di['db'])

Revert model data:

In [60]:
M.db.aom(adj.rc_pd(d0['db'](di['shock'].name), di['shock']))

### B. Collect and plot results

**Rebound rate over time -- Plastic**

In [61]:
%%capture
x = 'RCmandate'
di = shocks[x]
largeFont()
df = adj.rc_pd(di['Rbv_sum'], tPlot)
if slides:
    fig, ax = plt.subplots(1,1,figsize = (14,10))
else:
    fig, ax = plt.subplots(1,1,figsize = (8,5))
seaborn.lineplot(data = df, linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('Rebound rate');
ax.axhline(1, color = 'k', linewidth = 1, alpha = .5);
fig.tight_layout();
if slides:
    fig.savefig(os.path.join(d['figs'],f"GRSIntRC_{x}sum_Rbv_slides.png"),facecolor='#FAFAFA',edgecolor='k')
else:
    fig.savefig(os.path.join(d['figs'],f"GRSIntRC_{x}sum_Rbv.pdf"),edgecolor='k')

**Change in material throughout - across all materials:**

In [62]:
%%capture
largeFont()
x = 'RCmandate'
di = shocks[x]
df = pd.concat([di[f'ΔQ{k}/Q{k}_sum'].rename(f'$ΔQ_{k}/Q_{k}$') for k in ('v','r','t')], axis = 1)*100
df = adj.rc_pd(df, tPlot)
if slides:
    fig, ax = plt.subplots(1,1,figsize = (14,10))
else:
    fig, ax = plt.subplots(1,1,figsize = (8,5))
seaborn.lineplot(data = df, linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5);
fig.tight_layout();
if slides:
    fig.savefig(os.path.join(d['figs'],f"GRSIntRC_{x}sum_DiffMC_slides.png"),facecolor='#FAFAFA',edgecolor='k')
else:
    fig.savefig(os.path.join(d['figs'],f"GRSIntRC_{x}sum_DiffMC.pdf"),edgecolor='k')

**Change in material throughout - different material types:**

In [63]:
%%capture
largeFont()
x = 'RCmandate'
ti = tPlot_1
di = shocks[x]
df = pd.concat([di['ΔQv/Qv'].xs(ti), di['ΔQr/Qr'].xs(ti), di['ΔQt/Qt'].xs(ti), di['CMUR'].xs(ti)-d0['CMUR'].xs(ti)], axis = 1).set_axis(['$ΔQ_v/Q_v$',
                                                                                                                                    '$ΔQ_r/Q_r$',
                                                                                                                                    '$ΔQ_t/Q_t$',
                                                                                                                                    '$ΔCMUR$'], axis = 1) * 100
if slides:
    fig, ax = plt.subplots(1,1,figsize = (14,10))
else:
    fig, ax = plt.subplots(1,1,figsize = (8,5))
df[['$ΔQ_v/Q_v$','$ΔQ_r/Q_r$','$ΔQ_t/Q_t$']].plot.bar(ax = ax)
ax.set_xlabel('');
ax.set_ylabel('% change');
fig.tight_layout();
if slides:
    fig.savefig(os.path.join(d['figs'],f"GRSIntRC_{x}_DiffMC_slides.png"),facecolor='#FAFAFA',edgecolor='k')
else:
    fig.savefig(os.path.join(d['figs'],f"GRSIntRC_{x}_DiffMC.pdf"),edgecolor='k')

**Plot change in CMUR:**

In [64]:
%%capture
largeFont()
if slides:
    fig, ax = plt.subplots(1,1,figsize = (14,10))
else:
    fig, ax = plt.subplots(1,1,figsize = (8,5))
df['$ΔCMUR$'].plot.bar(ax=ax)
ax.set_xlabel('');
ax.set_ylabel('$Δ CMUR$');
fig.tight_layout();
if slides:
    fig.savefig(os.path.join(d['figs'],f"GRSIntRC_{x}_CMUR_slides.png"),facecolor='#FAFAFA',edgecolor='k')
else:
    fig.savefig(os.path.join(d['figs'],f"GRSIntRC_{x}_CMUR.pdf"),edgecolor='k')